# Train Naive Bayes Model



### 1. Load Processed Data



In [ ]:
from pathlib import Path

import pandas as pd

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "data").exists():
    ROOT_DIR = ROOT_DIR.parent

DATA_DIR = ROOT_DIR / "data" / "processed"

train_df = pd.read_csv(DATA_DIR / "train.csv")
validation_df = pd.read_csv(DATA_DIR / "validation.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

### 2. Check Split Balance



In [ ]:
for split_name, split_df in [
    ("Train", train_df),
    ("Validation", validation_df),
    ("Test", test_df),
]:
    print(f"\n{split_name}")
    print(split_df["spam"].value_counts().rename({0: "Not Spam", 1: "Spam"}))
    print((split_df["spam"].value_counts(normalize=True) * 100).round(2).rename({0: "Not Spam %", 1: "Spam %"}))

### 3. Separate Text and Labels



In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.naive_bayes import MultinomialNB

X_train = train_df["text"].astype(str)
y_train = train_df["spam"].astype(int)

X_val = validation_df["text"].astype(str)
y_val = validation_df["spam"].astype(int)

X_test = test_df["text"].astype(str)
y_test = test_df["spam"].astype(int)

### 4. Convert Email Text to Numeric Features


In [ ]:
vectorizer = CountVectorizer(stop_words="english")

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

print("Vocabulary size:", len(vectorizer.vocabulary_))
print("Train matrix shape:", X_train_vec.shape)
print("Validation matrix shape:", X_val_vec.shape)
print("Test matrix shape:", X_test_vec.shape)

### 5. Train the Model



In [ ]:
model = MultinomialNB()
model.fit(X_train_vec, y_train)

### 6. Evaluate Train, Validation, and Test Results



In [ ]:
LABELS = [0, 1]
TARGET_NAMES = ["Not Spam", "Spam"]


def evaluate_split(split_name, x_values, y_values):
    predictions = model.predict(x_values)
    accuracy = accuracy_score(y_values, predictions)
    report = classification_report(
        y_values,
        predictions,
        labels=LABELS,
        target_names=TARGET_NAMES,
        digits=4,
    )
    matrix = confusion_matrix(y_values, predictions, labels=LABELS)

    return (
        f"{split_name} Accuracy: {accuracy:.4f}\n"
        f"{split_name} Classification Report:\n{report}\n"
        f"{split_name} Confusion Matrix [[TN, FP], [FN, TP]]:\n{matrix}\n"
    )


report_sections = [
    evaluate_split("Train", X_train_vec, y_train),
    evaluate_split("Validation", X_val_vec, y_val),
    evaluate_split("Test", X_test_vec, y_test),
]
report_text = "\n".join(report_sections)
print(report_text)

### 7. Save the Model, Vectorizer, and Report



In [ ]:
import joblib

MODELS_DIR = ROOT_DIR / "models"
REPORTS_DIR = ROOT_DIR / "reports"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODELS_DIR / "naive_bayes_model.pkl"
vectorizer_path = MODELS_DIR / "vectorizer.pkl"
report_path = REPORTS_DIR / "classification_report.txt"

joblib.dump(model, model_path)
joblib.dump(vectorizer, vectorizer_path)
report_path.write_text(report_text, encoding="utf-8")

print("Saved model:", model_path)
print("Saved vectorizer:", vectorizer_path)
print("Saved report:", report_path)